# Phase 06A.04 — LoRA r16 two-stage full training
Selects training horizon on the train-side inner development split, then resets every state and retrains on all 1,192 training rows. Frozen validation is not loaded.

In [ ]:
import gc,json,os,sys
from pathlib import Path
PROJECT_ROOT=Path('/workspace/RoadBuddy'); SRC_DIR=PROJECT_ROOT/'src'
if str(SRC_DIR) not in sys.path: sys.path.insert(0,str(SRC_DIR))
os.chdir(PROJECT_ROOT)
from roadbuddy_common import *
from phase06a_common import *
seed_everything(SEED)

## Configuration
Full mode requires the completed inner-split protocol. Smoke outputs are isolated and cannot be used as final checkpoints.

In [ ]:
RUN_SCOPE = 'full'  # temporary execution copy: smoke | full
RANK=16; ALPHA=32; EPOCHS=2; GRADIENT_ACCUMULATION=16; LEARNING_RATE=1e-4; WEIGHT_DECAY=0.01; MAX_GRAD_NORM=1.0
SMOKE_TRAIN_LIMIT=20; SMOKE_DEV_LIMIT=20; SMOKE_MAX_STEPS=2
INNER_DIR=PROJECT_ROOT/'data/splits/phase06a_inner'; TRAIN_CSV=PROJECT_ROOT/'data/splits/phase01/train.csv'
OUTPUT_DIR=PROJECT_ROOT/'outputs/phase06a/lora_r16_training'/RUN_SCOPE; OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
protocol=Phase06AProtocol(run_scope=RUN_SCOPE)
assert all(path.is_file() for path in [INNER_DIR/'train_fit.csv',INNER_DIR/'inner_dev.csv',INNER_DIR/'checkpoint_protocol.json',TRAIN_CSV])

## Load train-side data only

In [ ]:
train_fit=pd.read_csv(INNER_DIR/'train_fit.csv'); inner_dev=pd.read_csv(INNER_DIR/'inner_dev.csv'); all_train=pd.read_csv(TRAIN_CSV)
assert not (set(train_fit.group_id.astype(str)) & set(inner_dev.group_id.astype(str)))
if RUN_SCOPE=='full': assert len(all_train)==EXPECTED_TRAIN_ROWS and all_train.group_id.nunique()==EXPECTED_TRAIN_GROUPS
calibration_train=train_fit.head(SMOKE_TRAIN_LIMIT) if RUN_SCOPE=='smoke' else train_fit
calibration_dev=inner_dev.head(SMOKE_DEV_LIMIT) if RUN_SCOPE=='smoke' else inner_dev
final_train=all_train.head(SMOKE_TRAIN_LIMIT) if RUN_SCOPE=='smoke' else all_train
checkpoint_protocol=json.loads((INNER_DIR/'checkpoint_protocol.json').read_text(encoding='utf-8'))

## Stage A — inner-development checkpoint selection

In [ ]:
stage_a_model,tokenizer=create_lora_model(RANK,ALPHA,training=True)
stage_a=run_lora_training_stage(stage_a_model,tokenizer,calibration_train,output_dir=OUTPUT_DIR/'stage_a_inner_selection',total_tile_budget=protocol.total_tile_budget,epochs=1 if RUN_SCOPE=='smoke' else EPOCHS,gradient_accumulation=GRADIENT_ACCUMULATION,learning_rate=LEARNING_RATE,weight_decay=WEIGHT_DECAY,max_grad_norm=MAX_GRAD_NORM,max_optimizer_steps=SMOKE_MAX_STEPS if RUN_SCOPE=='smoke' else None,evaluation_frame=calibration_dev,evaluation_limit=None,evaluation_steps=1 if RUN_SCOPE=='smoke' else checkpoint_protocol['eval_steps'],patience_evaluations=checkpoint_protocol['patience_evaluations'],seed=SEED)
locked_step=stage_a['best_step']; assert locked_step is not None and locked_step>0
save_json(OUTPUT_DIR/'best_step_decision.json',{'selection_data':'inner_dev_only','locked_optimizer_step':locked_step,'primary_metric':'accuracy','secondary_metric':'macro_f1','best_metrics':stage_a['best_metrics']})
del stage_a_model; gc.collect(); torch.cuda.empty_cache()

## Stage B — reset and retrain on all training rows

In [ ]:
seed_everything(SEED)
final_model,final_tokenizer=create_lora_model(RANK,ALPHA,training=True)
stage_b=run_lora_training_stage(final_model,final_tokenizer,final_train,output_dir=OUTPUT_DIR/'stage_b_final_retrain',total_tile_budget=protocol.total_tile_budget,epochs=max(EPOCHS,2),gradient_accumulation=GRADIENT_ACCUMULATION,learning_rate=LEARNING_RATE,weight_decay=WEIGHT_DECAY,max_grad_norm=MAX_GRAD_NORM,max_optimizer_steps=locked_step,scheduler_total_steps=stage_a['scheduler_total_steps'],evaluation_frame=None,evaluation_steps=checkpoint_protocol['eval_steps'],patience_evaluations=checkpoint_protocol['patience_evaluations'],seed=SEED)
assert stage_b['optimizer_steps']==locked_step and stage_b['scheduler_total_steps']==stage_a['scheduler_total_steps']
adapter_dir=OUTPUT_DIR/'final_adapter'; adapter_dir.mkdir(parents=True,exist_ok=True)
final_model.save_pretrained(adapter_dir); final_tokenizer.save_pretrained(adapter_dir)
config={**protocol.to_dict(),'rank':RANK,'alpha':ALPHA,'alpha_over_rank':ALPHA/RANK,'dropout':0.05,'targets':['q_proj','k_proj','v_proj','o_proj'],'learning_rate':LEARNING_RATE,'weight_decay':WEIGHT_DECAY,'gradient_accumulation':GRADIENT_ACCUMULATION,'locked_optimizer_step':locked_step,'scheduler_total_steps':stage_a['scheduler_total_steps'],'selection_data':'inner_dev_only','final_training_rows':len(final_train)}
save_json(OUTPUT_DIR/'config.json',config)

## Adapter reload and provenance gates

In [ ]:
from peft import PeftModel
del final_model; gc.collect(); torch.cuda.empty_cache()
reload_base,reload_tokenizer=load_model_and_tokenizer(training=False,attn_implementation=protocol.attention_implementation)
reloaded=PeftModel.from_pretrained(reload_base,adapter_dir); reloaded.eval(); assert reloaded.peft_config
del reloaded,reload_base; gc.collect(); torch.cuda.empty_cache()
artifacts=[OUTPUT_DIR/'config.json',OUTPUT_DIR/'best_step_decision.json',OUTPUT_DIR/'stage_a_inner_selection/training_history.csv',OUTPUT_DIR/'stage_a_inner_selection/training_result.json',OUTPUT_DIR/'stage_b_final_retrain/training_history.csv',OUTPUT_DIR/'stage_b_final_retrain/training_result.json',adapter_dir/'adapter_config.json',adapter_dir/'adapter_model.safetensors']
manifest=write_run_manifest(OUTPUT_DIR/'run_manifest.json',config=config,artifacts=artifacts)
status='complete' if RUN_SCOPE=='full' and len(final_train)==EXPECTED_TRAIN_ROWS else 'smoke_complete'
save_json(OUTPUT_DIR/'PHASE06A_04_STATUS.json',{'phase':'06A.04','status':status,'run_scope':RUN_SCOPE,'locked_optimizer_step':locked_step,'adapter_sha256':sha256_file(adapter_dir/'adapter_model.safetensors'),'final_training_rows':len(final_train)})
display({'stage_a':stage_a,'stage_b':stage_b,'status':status})

## Interpretation constraint
Inner-development metrics select training horizon only. This notebook produces no frozen-validation estimate and cannot establish that LoRA improves over zero-shot.